# Product Page Info — Bath Bombs

Exploratory analysis and visualizations for `product_page_info_bath_bombs_all.csv`.

> Run all cells top-to-bottom. Data path: `../input/product_page_info_bath_bombs_all.csv`

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)

DATA_PATH = Path("../input/product_page_info_bath_bombs_all.csv")
df_raw = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Loaded {len(df_raw):,} rows and {df_raw.shape[1]} columns from {DATA_PATH}")

## 1. Dataset overview

In [ ]:
display(df_raw.head())
display(df_raw.info())
display(df_raw.describe(include="all").T)

In [ ]:
missing_pct = (df_raw.isna().mean() * 100).sort_values(ascending=False)
missing_df = missing_pct.to_frame("missing_pct")

fig, ax = plt.subplots(figsize=(8, 9))
sns.barplot(
    data=missing_df,
    y=missing_df.index,
    x="missing_pct",
    hue=missing_df.index,
    dodge=False,
    legend=False,
    palette="Blues_r",
    ax=ax,
)
ax.set_title("Missing values by column (%)")
ax.set_xlabel("Missing %")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

display(missing_df.head(15))

## 2. Clean sentinel values

Several columns use `"Not Found"`, empty strings, or list-like strings. Convert those to missing values and parse numeric fields.

In [ ]:
SENTINELS = {"Not Found", "not found", "", "[]", "nan", "NaN"}


def clean_sentinels(series: pd.Series) -> pd.Series:
    cleaned = series.replace(list(SENTINELS), np.nan)
    if cleaned.dtype == object:
        cleaned = cleaned.replace(r"^\s*$", np.nan, regex=True)
    return cleaned


def to_numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(clean_sentinels(series), errors="coerce")


df = df_raw.copy()
for col in df.select_dtypes(include="object").columns:
    df[col] = clean_sentinels(df[col])

df["price_num"] = to_numeric(df["price"])
df["rating_num"] = to_numeric(df["rating"])
df["review_num_int"] = to_numeric(df["review_num"])
df["prime_flag"] = df["prime"].fillna(0).astype(int)

df["first_available_dt"] = pd.to_datetime(df["first_available"], errors="coerce")
df["first_available_year"] = df["first_available_dt"].dt.year

df["has_feature"] = df["feature"].notna().astype(int)
df["has_description"] = df["product_description"].notna().astype(int)

numeric_cols = [
    "price_num",
    "rating_num",
    "review_num_int",
    "image_num",
    "number_of_items",
    "variation_quantity",
    "other_seller_num",
    "per_unit_price",
    "unit_num",
    "label_unit_num",
    "prime_flag",
]

print("Parsed numeric coverage:")
for col in numeric_cols:
    print(f"  {col:18s}: {df[col].notna().sum():>6,} non-null ({df[col].notna().mean()*100:5.1f}%)")

## 3. Price

In [ ]:
priced = df["price_num"].dropna()
print(f"Products with a listed price: {len(priced):,} ({len(priced)/len(df)*100:.1f}%)")
print(priced.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]).round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(priced, bins=50, kde=True, ax=axes[0], color="#4C72B0")
axes[0].set_title("Price distribution")
axes[0].set_xlabel("Price (USD)")

sns.histplot(priced[priced <= 50], bins=40, kde=True, ax=axes[1], color="#55A868")
axes[1].set_title("Price distribution (<= $50)")
axes[1].set_xlabel("Price (USD)")

plt.tight_layout()
plt.show()

## 4. Ratings and reviews

In [ ]:
rated = df["rating_num"].dropna()
reviewed = df["review_num_int"].dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

rating_counts = rated.value_counts().sort_index()
sns.barplot(x=rating_counts.index, y=rating_counts.values, ax=axes[0], color="#C44E52")
axes[0].set_title("Rating distribution")
axes[0].set_xlabel("Star rating")
axes[0].set_ylabel("Count")

review_cap = reviewed[reviewed <= 500]
sns.histplot(review_cap, bins=50, ax=axes[1], color="#8172B2")
axes[1].set_title("Review count distribution (<= 500 reviews)")
axes[1].set_xlabel("Number of reviews")

plt.tight_layout()
plt.show()

print(f"Products with ratings: {len(rated):,}")
print(f"Products with at least 1 review: {(reviewed > 0).sum():,}")

In [ ]:
scatter_df = df[["price_num", "rating_num", "review_num_int"]].dropna()
scatter_df = scatter_df[scatter_df["review_num_int"] <= 1000]

fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(
    data=scatter_df,
    x="price_num",
    y="rating_num",
    size="review_num_int",
    sizes=(20, 300),
    alpha=0.5,
    ax=ax,
)
ax.set_title("Price vs. rating (bubble size = review count)")
ax.set_xlim(0, 60)
plt.tight_layout()
plt.show()

## 5. Brands

In [ ]:
top_brands = df["brand"].value_counts().head(20)

fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(x=top_brands.values, y=top_brands.index, hue=top_brands.index, dodge=False, legend=False, ax=ax)
ax.set_title("Top 20 brands by product count")
ax.set_xlabel("Number of products")
plt.tight_layout()
plt.show()

print(f"Unique brands: {df['brand'].nunique():,}")
print(f"Products missing brand: {df['brand'].isna().sum():,}")

## 6. Stock status and Prime

In [ ]:
def simplify_stock_status(status: str) -> str:
    if pd.isna(status):
        return "Missing"
    status_lower = status.lower()
    if "in stock" in status_lower and "only" not in status_lower:
        return "In stock"
    if "only" in status_lower and "left" in status_lower:
        return "Low stock"
    if "currently unavailable" in status_lower:
        return "Unavailable"
    if "stock status not found" in status_lower:
        return "Unknown"
    return "Other"


df["stock_group"] = df["stock_status"].apply(simplify_stock_status)
stock_counts = df["stock_group"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

stock_counts.plot(kind="bar", ax=axes[0], color="#4C72B0")
axes[0].set_title("Stock status (grouped)")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=30)

prime_counts = df["prime_flag"].value_counts().rename({0: "Not Prime", 1: "Prime"})
axes[1].pie(
    prime_counts,
    labels=prime_counts.index,
    autopct="%1.1f%%",
    startangle=90,
    colors=["#DD8452", "#4C72B0"],
)
axes[1].set_title("Prime eligibility")

plt.tight_layout()
plt.show()

## 7. Pack size and unit fields

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

items = df["number_of_items"].dropna()
items_cap = items[items <= 24]
sns.countplot(x=items_cap, ax=axes[0], color="#55A868")
axes[0].set_title("Number of items (<= 24)")
axes[0].tick_params(axis="x", rotation=45)

variations = df["variation_quantity"].dropna()
sns.countplot(x=variations, ax=axes[1], color="#C44E52")
axes[1].set_title("Variation quantity")
axes[1].tick_params(axis="x", rotation=0)

label_units = df["label_unit"].value_counts().head(12)
sns.barplot(x=label_units.values, y=label_units.index, hue=label_units.index, dodge=False, legend=False, ax=axes[2])
axes[2].set_title("Top label units")
axes[2].set_xlabel("Count")

plt.tight_layout()
plt.show()

## 8. Images and listing completeness

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(data=df, x="image_num", ax=axes[0], color="#8172B2")
axes[0].set_title("Number of product images")

completeness = pd.DataFrame(
    {
        "Has feature text": df["has_feature"].mean() * 100,
        "Has description": df["has_description"].mean() * 100,
        "Has price": df["price_num"].notna().mean() * 100,
        "Has rating": df["rating_num"].notna().mean() * 100,
        "Has brand": df["brand"].notna().mean() * 100,
        "Has dimensions": df["product_dimensions"].notna().mean() * 100,
    },
    index=["coverage_pct"],
).T.sort_values("coverage_pct")

sns.barplot(
    data=completeness.reset_index(names="field"),
    y="field",
    x="coverage_pct",
    hue="field",
    dodge=False,
    legend=False,
    ax=axes[1],
    palette="viridis",
)
axes[1].set_title("Field coverage (% of products)")
axes[1].set_xlabel("Coverage %")

plt.tight_layout()
plt.show()

display(completeness.round(1))

## 9. First available year

In [ ]:
year_counts = df["first_available_year"].dropna().astype(int).value_counts().sort_index()
recent_years = year_counts[year_counts.index >= 2000]

fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(x=recent_years.index, y=recent_years.values, marker="o", ax=ax, color="#4C72B0")
ax.set_title("Products by first-available year (2000+)")
ax.set_xlabel("Year")
ax.set_ylabel("Number of products")
plt.tight_layout()
plt.show()

## 10. Correlations among numeric fields

In [ ]:
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation matrix (numeric fields)")
plt.tight_layout()
plt.show()

## 11. Quick text-field peek

In [ ]:
text_fields = ["title", "feature", "product_description", "best_seller_rank", "stock_status"]

for field in text_fields:
    non_null = df[field].dropna()
    print(f"\n=== {field} ===")
    print(f"Non-null: {len(non_null):,} | Unique: {non_null.nunique():,}")
    if len(non_null):
        sample = non_null.sample(min(3, len(non_null)), random_state=42)
        for i, value in enumerate(sample, start=1):
            preview = str(value).replace("\n", " ")[:180]
            print(f"  {i}. {preview}{'...' if len(str(value)) > 180 else ''}")